# Pandas Data Cleaning and Summary Statistics

**Topic:** Pandas data cleaning &nbsp;·&nbsp; **Difficulty:** Easy

## The data (created in the next cell)

- `df`: a pandas DataFrame of temperature readings with columns `sensor_id`, `timestamp`, `temp_c`, `humidity`.
- The log was corrupted: some rows are **exact duplicates**, and some rows have an **invalid temperature** (either `NaN` or the sentinel value `-999.0`).
- Rows are shuffled, so bad rows are mixed in everywhere.

## Your task

Clean the log using exactly these rules, in this order:

1. Drop exact duplicate rows (keep the first occurrence).
2. Drop every row whose `temp_c` is `NaN` or equal to `-999.0`.

Then compute three quantities from the cleaned data and store them in a dict named `answers`:

```python
answers = {
    "n_clean_rows": ...,   # int: rows remaining after cleaning
    "mean_temp_c": ...,    # float: mean of temp_c, any precision
    "hottest_sensor": ..., # str: sensor_id with the highest mean temp_c
}
```

## Evaluation

The evaluation cell checks all three answers (floats are checked with a small tolerance). All three must be correct to pass.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# --- build the true, clean log (used to grade you, so do not peek at _expected) ---
n = 900
sensors = [f"S{i:02d}" for i in range(8)]
base_temp = {s: rng.uniform(16, 30) for s in sensors}
rows = []
for i in range(n):
    s = sensors[rng.integers(len(sensors))]
    rows.append({
        "sensor_id": s,
        "timestamp": pd.Timestamp("2026-05-01") + pd.Timedelta(minutes=int(i * 7 + rng.integers(0, 5))),
        "temp_c": round(base_temp[s] + rng.normal(0, 2.0), 2),
        "humidity": round(rng.uniform(35, 85), 1),
    })
clean = pd.DataFrame(rows).drop_duplicates()

_by_sensor = clean.groupby("sensor_id")["temp_c"].mean()
_expected = {
    "n_clean_rows": int(len(clean)),
    "mean_temp_c": float(clean["temp_c"].mean()),
    "hottest_sensor": str(_by_sensor.idxmax()),
}

# --- corrupt it: duplicated rows + rows with invalid temperatures ---
dupes = clean.sample(140, random_state=1)
bad = clean.sample(120, random_state=2).copy()
bad["temp_c"] = np.where(rng.random(len(bad)) < 0.5, np.nan, -999.0)
bad["timestamp"] = bad["timestamp"] + pd.Timedelta(seconds=13)  # not duplicates of clean rows
df = (
    pd.concat([clean, dupes, bad], ignore_index=True)
    .sample(frac=1, random_state=3)
    .reset_index(drop=True)
)

print(f"df shape: {df.shape}")
df.head()


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Clean the DataFrame `df` created above and store the `answers` dict
# the task section asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your answers to the truth
# =====================================================================
if "answers" not in dir():
    raise NameError("Define the `answers` dict in the solution cell before running this cell.")

_checks = [
    ("n_clean_rows", lambda a, e: int(a) == int(e)),
    ("mean_temp_c", lambda a, e: abs(float(a) - float(e)) < 0.05),
    ("hottest_sensor", lambda a, e: str(a) == str(e)),
]
_ok = 0
print("=" * 52)
for _key, _cmp in _checks:
    if _key not in answers:
        print(f"  {_key:<16} MISSING from answers")
        continue
    _good = _cmp(answers[_key], _expected[_key])
    _ok += _good
    print(f"  {_key:<16} yours={answers[_key]!r:<24} {'CORRECT' if _good else 'WRONG'}")
print("=" * 52)
print(f"Score: {_ok}/3")
print("PASSED. Nice work!" if _ok == 3 else "NOT PASSED yet. Re-read the cleaning rules.")


## Solution

Apply the two cleaning rules in order: `drop_duplicates()` then a mask that removes `NaN` and `-999.0` temperatures, and the three answers are one-liners on the cleaned frame.

```python
cleaned = df.drop_duplicates()
cleaned = cleaned[cleaned["temp_c"].notna() & (cleaned["temp_c"] != -999.0)]

answers = {
    "n_clean_rows": int(len(cleaned)),
    "mean_temp_c": float(cleaned["temp_c"].mean()),
    "hottest_sensor": str(cleaned.groupby("sensor_id")["temp_c"].mean().idxmax()),
}
print(answers)
```
